# Module 3 — Enriching the Graph with Structured Data

**The gap, from Module 2:** `Company → Document → Chunk` only holds what's inside the 10-Ks.
Officers are named there, but their career history is "incorporated by reference" to the proxy
statement — a document never ingested. No amount of retrying in the agentic loop fixes a graph
coverage gap; it needs more graph.

**What we build:** two structured data loaders — Wikidata (executives/board, with career
history) and the NYT Article Search API (news) — writing `Person` and `Article` nodes into the
same graph, plus a fourth retrieval tool, `get_executives`, that the Module 2 agent picks up
automatically.

**New components introduced:**
- `enrichment.loaders` — `load_executives` (Wikidata SPARQL) and `load_news` (NYT Article Search API)
- `enrichment.graph_writer` — `write_executives`/`write_news`, upserting `Person`/`Article` and
  `ROLE_AT`/`MENTIONED_IN`
- `ingestion.schema.apply_enrichment_schema()` — constraints + fulltext index for `Person`,
  `Event`, `Article`
- `retrieval.graph_nav.get_executives` + `agent.tools.get_executives` — a new tool for the
  Module 2 retrieval loop, plus an updated strategy prompt that knows when to reach for it

> Constraints are idempotent (`IF NOT EXISTS`) and every writer `MERGE`s, so all cells below are
> safe to re-run.


## 1. Recap — Where Module 2 Left Off

Module 2 closed by asking the agent a people question the document graph can't answer: *which
other companies have 3M's current executive officers previously worked at or served as directors
of?* `semantic_search`/`fulltext_search` could only surface the 10-K's own text, which names the
officers but explicitly defers their career history to 3M's proxy statement — a document outside
this corpus. We'll come back to that exact question at the end of this notebook, once the graph
actually holds the answer.


## 2. Schema — `Person`, `Event`, `Article`

Same pattern as Module 1's `apply_basic_schema()`: uniqueness constraints plus a fulltext index,
defined in `ingestion.schema` as `CONSTRAINTS_M3`/`INDEXES_M3` and applied idempotently. `Event`
gets a constraint here too (a placeholder for a richer corporate-events model), but this module
only populates `Person` and `Article` — event ingestion isn't part of this course.


In [ ]:
from financial_advisor.ingestion.schema import apply_enrichment_schema

apply_enrichment_schema()


## 3. Executives and Board Members — Wikidata

`load_executives(company_id)` runs a SPARQL query against Wikidata for officers (CEO `P169`,
chairperson `P488`) and board members (`P3320`), then augments each person with a Wikipedia
summary and their own career history (`P108` employer claims) — this is the data Module 2
needed. Board members are scoped to `FILING_YEAR = 2018` so the roster matches the 10-Ks already
in the graph; the CEO/chairperson title is cross-checked against the person's own tenure history
for the same reason (Wikidata's org-level claims only reflect the *current* holder, with no
dates). Rate-limited to Wikidata's public SPARQL endpoint, so this takes a little while per
company.


In [ ]:
from financial_advisor.enrichment.loaders import load_executives

executives_by_company = {}
for company_id in ["3M", "APPLE"]:
    executives = load_executives(company_id)
    executives_by_company[company_id] = executives
    print(f"{company_id}: {len(executives)} executives/board members")


In [ ]:
import json

sample = executives_by_company["3M"][0]
print(json.dumps(sample, indent=2))


## 4. Writing Executives to the Graph

`write_executives` upserts one `Person` node per person (`id`, `name`, `bio`) and `ROLE_AT`
edges for every title — both at the target company and at each `career_history` employer.
Employers outside the two companies ingested in Module 1 land as lightweight stub `Company`
nodes (`stub: true`); we only know their name here, not their filings.


In [ ]:
from financial_advisor.enrichment.graph_writer import write_executives

for company_id, executives in executives_by_company.items():
    write_executives(company_id, executives)


In [ ]:
from financial_advisor.services.neo4j_service import neo4j_service

rows = neo4j_service.run_query(
    "MATCH (p:Person)-[r:ROLE_AT]->(c:Company) "
    "RETURN c.id AS company, c.stub AS is_stub, p.name AS person, r.title AS title "
    "ORDER BY company, person LIMIT 15"
)
for row in rows:
    print(row)


## 5. News — NYT Article Search

`load_news(company_id, start_date, end_date)` queries the NYT Article Search API and `write_news`
upserts `Article` nodes linked to the company via `MENTIONED_IN`. One honest caveat: this is a
*mentions* relationship from a general-purpose news search, not an "this article is about the
company" classifier — a company as large as 3M or Apple shows up in plenty of daily
market-roundup articles that only name it once, in passing, alongside dozens of others. That's a
different precision/recall trade-off than the Wikidata lookup above, and worth keeping in mind
when a retrieval tool eventually searches `Article.text` too.


In [ ]:
from financial_advisor.enrichment.graph_writer import write_news
from financial_advisor.enrichment.loaders import load_news

for company_id in ["3M", "APPLE"]:
    articles = load_news(company_id, start_date="2018-01-01", end_date="2018-12-31")
    write_news(company_id, articles)
    print(f"{company_id}: {len(articles)} articles")


In [ ]:
rows = neo4j_service.run_query(
    "MATCH (c:Company)-[:MENTIONED_IN]->(a:Article) "
    "RETURN c.id AS company, a.published_at AS published_at, a.title AS title "
    "ORDER BY published_at LIMIT 10"
)
for row in rows:
    print(row)


## 6. A New Retrieval Tool — `get_executives`

`retrieval.graph_nav.get_executives` is a structured Cypher lookup, not a vector/fulltext search:
for a company, it returns each `Person`'s roles there plus their `career_history` at every
*other* company they have a `ROLE_AT` edge to. Wrapped as `agent.tools.get_executives` and added
to `agent.tools.TOOLS`, it's now a fourth option the Module 2 strategy agent can pick —
`agent.prompts.STRATEGY_SYSTEM_PROMPT` was updated to describe it, since tool binding alone
doesn't tell the model *when* to prefer it over free-text search over the filings.

No changes were needed to `agent.graph`, `agent.nodes`, or `agent.state` — `call_tools_node`
already dispatches generically over `TOOLS_BY_NAME`. The one thing that did need a fix:
`evaluate_retrieval`'s grading prompt assumed every retrieved item was a document chunk
(`doc_id`/`text`/`pages`); it now renders non-chunk results like this one as labeled fact
records instead.


In [ ]:
from financial_advisor.agent.tools import get_executives

hits = get_executives.invoke({"company_id": "3M"})
print(f"get_executives('3M') -> {len(hits)} people\n")
for h in hits[:3]:
    roles = [r["title"] for r in h["roles"]]
    print(f"  {h['name']}  roles={roles}")
    if h["career_history"]:
        history = [(c["company"], c["title"]) for c in h["career_history"]]
        print(f"    career_history: {history}")


## 7. Closing the Loop — the Module 2 Struggle Question, Revisited

Same agent (`agent.graph.build_agent()`), same question, same retrieval loop — the only things
that changed are what's in the graph and which tools the strategy agent can reach for.


In [ ]:
from financial_advisor.agent.graph import build_agent
from financial_advisor.agent.state import initial_state

STRUGGLE_QUESTION = (
    "Which other companies have 3M's current executive officers previously worked at or "
    "served as directors of?"
)

agent = build_agent()
result = agent.invoke(initial_state(STRUGGLE_QUESTION), {"recursion_limit": 50})
print(
    f"\n[{result['retrieval_iterations']} retrieval round(s), "
    f"{result['answer_attempts']} answer attempt(s), "
    f"{len(result['retrieved_chunks'])} item(s) retrieved]"
)
print(f"\nA: {result['answer']}")


Compare this to Module 2, where the same question exhausted every retry with the agent honestly
reporting the knowledge wasn't there. Now it isn't — but notice *how* it got there:

- **Iteration 1** went straight to `get_executives('3M')` — the strategy prompt update worked,
  the agent reached for the new structured tool first instead of falling back to free-text
  search. It surfaced Mike Roman's career history (Hughes Aircraft Company) instantly and
  precisely, no LLM extraction guesswork involved.
- **`evaluate_retrieval` correctly judged that wasn't enough.** Wikidata's officer query is
  strong on the CEO and the board, but SEC disclosure rules require naming *all* executive
  officers (CFO, other named execs) — a roster Wikidata mostly doesn't carry. So the loop fell
  back to `semantic_search`/`fulltext_search` for three more rounds, which is exactly the
  Module 2 behavior, just now layered on top of a first structured hit instead of starting cold.
- Those later rounds found something worth noting: the 10-K's own Item 401 "business experience"
  disclosure does list a few years of prior employers for several named officers — a fact the
  original graph coverage gap analysis didn't anticipate. Real filings vary in how much they
  actually defer to the proxy.
- The answer accepted on the first attempt, cites `doc_id`/`chunk_id` throughout, and — this is
  the important part — explicitly names the five officers it still has no external-employer data
  for, rather than silently omitting them. That's the graph's actual current coverage, honestly
  reported.

The remaining gap (those five officers) is a real one: `get_executives` only knows what
Wikidata's community-maintained officer/board data covers, and that isn't the same as SEC Item
401(b) disclosure. Closing it would mean extracting the 10-K's own executive-officer bio table
directly — which is exactly what Module 4 does next.


## 8. What's Next

Module 4 runs LLM extraction over every chunk to pull out dynamically-typed entities and
relationships the fixed schema doesn't anticipate — including, as section 7 just showed, the
kind of executive-officer business-experience detail buried in a 10-K's own text that a
structured external source like Wikidata doesn't fully cover.
